# Refugee Allocation — Greedy Nearest-Shelter Algorithm

Distributes earthquake-predicted refugees from each village to the nearest shelter
with remaining capacity.

**Overflow policy:** if all shelters are at capacity, the remainder goes to the
least-full shelter and is flagged `overflow=True`.

## Cell 1 — Imports + all definitions

In [ ]:
import json
import math
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Optional

# ── Data classes ──────────────────────────────────────────────────────────────

@dataclass
class Shelter:
    id: str           # e.g. "SA100-0002"  (string, not integer)
    name: str
    lat: float
    lon: float
    capacity: int
    city: str
    district: str
    village: str
    address: str
    allocated: int = field(default=0, repr=False)

    @property
    def remaining(self) -> int:
        return self.capacity - self.allocated

    @property
    def utilisation(self) -> float:
        if self.capacity == 0:
            return float('inf')
        return self.allocated / self.capacity


@dataclass
class Flow:
    from_zone: str        # "行政區-里名"
    to_shelter: str       # shelter name
    to_shelter_id: str    # e.g. "SA100-0002"
    people: int
    overflow: bool = False

    def to_dict(self) -> dict:
        return {
            'from_zone':     self.from_zone,
            'to_shelter':    self.to_shelter,
            'to_shelter_id': self.to_shelter_id,
            'people':        self.people,
            'overflow':      self.overflow,
        }


# ── Helpers ───────────────────────────────────────────────────────────────────

def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in kilometres."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def load_shelters(csv_path: str) -> List[Shelter]:
    """
    Load shelter.csv.
    Handles:
      - BOM header (encoding='utf-8-sig')
      - String IDs like 'SA100-0002' (no int() conversion)
      - Embedded newlines in id cells (strip)
      - Non-numeric capacity rows, e.g. '搬遷中' (skipped with a warning)
    """
    df = pd.read_csv(csv_path, encoding='utf-8-sig', dtype=str)
    shelters = []
    skipped  = []

    for _, row in df.iterrows():
        sid = str(row['id']).strip()
        cap_raw = str(row['capacity']).strip()
        try:
            cap = int(cap_raw)
        except ValueError:
            skipped.append(f"{sid} ({row['name'].strip()}) — capacity='{cap_raw}'")
            continue

        shelters.append(Shelter(
            id=sid,
            name=str(row['name']).strip(),
            lat=float(row['lat']),
            lon=float(row['lon']),
            capacity=cap,
            city=str(row.get('city', '')).strip(),
            district=str(row.get('district', '')).strip(),
            village=str(row.get('village', '')).strip(),
            address=str(row.get('address', '')).strip(),
        ))

    if skipped:
        print(f"  Skipped {len(skipped)} shelter(s) with non-numeric capacity:")
        for s in skipped:
            print(f"   {s}")

    return shelters


def load_villages(csv_path: str) -> pd.DataFrame:
    """
    Load village.csv.
    Columns: id1, id, district, village, lat, lon, population, address(office)
    Returns a cleaned DataFrame.
    """
    df = pd.read_csv(csv_path, encoding='utf-8-sig', dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df['lat']        = df['lat'].astype(float)
    df['lon']        = df['lon'].astype(float)
    df['population'] = pd.to_numeric(df['population'], errors='coerce').fillna(0).astype(int)
    return df


# ── Greedy allocation ─────────────────────────────────────────────────────────

def greedy_allocate(
    impact_data: List[dict],
    shelters: List[Shelter],
) -> List[Flow]:
    """
    Distribute refugees to shelters using a greedy nearest-first strategy.

    impact_data keys required: '行政區', '里名', 'lat', 'lon', '預估避難人數'
    Shelter.allocated is updated in-place.
    """
    flows: List[Flow] = []
    # Largest zones pick first — prevents big groups being left with distant shelters
    sorted_zones = sorted(impact_data, key=lambda z: z['預估避難人數'], reverse=True)

    for zone in sorted_zones:
        remaining = int(zone['預估避難人數'])
        if remaining <= 0:
            continue

        label   = f"{zone['行政區']}-{zone['里名']}"
        by_dist = sorted(
            shelters,
            key=lambda s: haversine(float(zone['lat']), float(zone['lon']), s.lat, s.lon)
        )

        for shelter in by_dist:
            if remaining <= 0:
                break
            avail = shelter.remaining
            if avail <= 0:
                continue
            send = min(remaining, avail)
            shelter.allocated += send
            remaining -= send
            flows.append(Flow(
                from_zone=label, to_shelter=shelter.name,
                to_shelter_id=shelter.id, people=send, overflow=False,
            ))

        # Soft overflow: all shelters full, assign to least-full and flag it
        if remaining > 0:
            fallback = min(by_dist, key=lambda s: s.utilisation)
            fallback.allocated += remaining
            flows.append(Flow(
                from_zone=label, to_shelter=fallback.name,
                to_shelter_id=fallback.id, people=remaining, overflow=True,
            ))

    return flows


print(' All definitions loaded.')

✅ All definitions loaded.


## Cell 2 — Load shelter & village data

In [ ]:
SHELTER_CSV = '/content/sample_data/shelter.csv'
VILLAGE_CSV = '/content/sample_data/village.csv'

shelters = load_shelters(SHELTER_CSV)
villages = load_villages(VILLAGE_CSV)

total_cap = sum(s.capacity for s in shelters)
print(f'Shelters loaded : {len(shelters)}   total capacity = {total_cap:,}')
print(f'Villages loaded : {len(villages)}')
print()
print('Shelter sample:')
display(pd.DataFrame([
    {
        'id': s.id, 'name': s.name, 'district': s.district,
        'capacity': s.capacity, 'lat': s.lat, 'lon': s.lon,
    } for s in shelters[:5]]))

⚠️  Skipped 1 shelter(s) with non-numeric capacity:
   SA105-1029 (臺北市立圖書館松山分館(暫停用)) — capacity='搬遷中'
Shelters loaded : 419   total capacity = 976,420
Villages loaded : 456

Shelter sample:


,id,name,district,capacity,lat,lon
0,SA100-0002,臺北市立螢橋國民中學,中正區,267,25.016335,121.530379
1,SA100-0003,臺北市立大學附設實驗國民小學,中正區,973,25.040125,121.516422
2,SA100-0004,臺北市立弘道國民中學,中正區,733,25.041697,121.516246
3,SA100-0005,二二八和平公園,中正區,5424,25.041076,121.514937
4,SA100-0006,臺北市中正運動中心,中正區,483,25.038166,121.521832


## Cell 3 — Earthquake impact data

**Paste your real output here, or load it from a JSON file.**

The sample below is generated from the actual village coordinates in `village.csv`
so you can run everything end-to-end immediately.

In [ ]:
'''
# ── Option A: paste your simulation output directly ───────────────────────────
impact_data: List[dict] = [
    # Replace or extend with your real data. Required keys:
    # '行政區', '里名', 'lat', 'lon',
    # '震央距離_km', '預估PGA', '預估震度', '預估避難人數'

    # ── 信義區 ────────────────────────────────────────────────────────────────
    {'行政區':'信義區','里名':'興雅里','lat':25.0423,'lon':121.5666,
     '震央距離_km':3.1,'預估PGA':0.88,'預估震度':6,'預估避難人數':896},
    {'行政區':'信義區','里名':'廣居里','lat':25.0419,'lon':121.5664,
     '震央距離_km':3.3,'預估PGA':0.84,'預估震度':6,'預估避難人數':1263},
    {'行政區':'信義區','里名':'國業里','lat':25.0415,'lon':121.5702,
     '震央距離_km':3.8,'預估PGA':0.79,'預估震度':6,'預估避難人數':1265},
    {'行政區':'信義區','里名':'大仁里','lat':25.0371,'lon':121.5746,
     '震央距離_km':4.2,'預估PGA':0.73,'預估震度':6,'預估避難人數':1431},
    {'行政區':'信義區','里名':'安康里','lat':25.0247,'lon':121.5786,
     '震央距離_km':6.1,'預估PGA':0.55,'預估震度':5,'預估避難人數':1894},

    # ── 大安區 ────────────────────────────────────────────────────────────────
    {'行政區':'大安區','里名':'通化里','lat':25.0333,'lon':121.5526,
     '震央距離_km':5.4,'預估PGA':0.60,'預估震度':5,'預估避難人數':993},
    {'行政區':'大安區','里名':'大學里','lat':25.0323,'lon':121.5448,
     '震央距離_km':6.2,'預估PGA':0.52,'預估震度':5,'預估避難人數':1526},
    {'行政區':'大安區','里名':'龍安里','lat':25.0287,'lon':121.535,
     '震央距離_km':7.0,'預估PGA':0.46,'預估震度':5,'預估避難人數':1398},

    # ── 松山區 ────────────────────────────────────────────────────────────────
    {'行政區':'松山區','里名':'龍田里','lat':25.0525,'lon':121.5628,
     '震央距離_km':4.9,'預估PGA':0.64,'預估震度':5,'預估避難人數':1946},
    {'行政區':'松山區','里名':'民有里','lat':25.0547,'lon':121.5654,
     '震央距離_km':5.1,'預估PGA':0.61,'預估震度':5,'預估避難人數':1497},
    {'行政區':'松山區','里名':'復盛里','lat':25.0535,'lon':121.563,
     '震央距離_km':5.3,'預估PGA':0.59,'預估震度':5,'預估避難人數':1765},

    # ── 中正區 ────────────────────────────────────────────────────────────────
    {'行政區':'中正區','里名':'南福里','lat':25.0405,'lon':121.5226,
     '震央距離_km':8.5,'預估PGA':0.40,'預估震度':4,'預估避難人數':2090},
    {'行政區':'中正區','里名':'新營里','lat':25.0327,'lon':121.505,
     '震央距離_km':9.8,'預估PGA':0.33,'預估震度':4,'預估避難人數':1490},
]
'''

# ── Option B: load from a JSON file saved by your simulation ─────────────────
with open('earthquake_simulation_result.json', 'r', encoding='utf-8') as f:
     impact_data = json.load(f)

total_refugees = sum(int(z['預估避難人數']) for z in impact_data)
print(f'Impact zones    : {len(impact_data)}')
print(f'Total refugees  : {total_refugees:,}')
print(f'Total capacity  : {sum(s.capacity for s in shelters):,}')

Impact zones    : 13
Total refugees  : 19,454
Total capacity  : 976,420


## Cell 4 — Run the algorithm

In [ ]:
# Reset allocations in case you re-run this cell
for s in shelters:
    s.allocated = 0

flows = greedy_allocate(impact_data, shelters)

normal_flows   = [f for f in flows if not f.overflow]
overflow_flows = [f for f in flows if f.overflow]
total_alloc    = sum(f.people for f in flows)
total_overflow = sum(f.people for f in overflow_flows)

print(f'Total flows      : {len(flows)}')
print(f'  Normal         : {len(normal_flows)}  →  {sum(f.people for f in normal_flows):,} people')
print(f'  Overflow ⚠️    : {len(overflow_flows)}  →  {total_overflow:,} people')
print(f'Total allocated  : {total_alloc:,} / {total_refugees:,}')

if overflow_flows:
    print('\n⚠️  Overflow detail:')
    for f in overflow_flows:
        print(f'  {f.from_zone}  →  {f.to_shelter}  (+{f.people:,} over capacity)')

Total flows      : 32
  Normal         : 32  →  19,454 people
  Overflow ⚠️    : 0  →  0 people
Total allocated  : 19,454 / 19,454


## Cell 5 — Flow table

In [ ]:
flows_df = pd.DataFrame([f.to_dict() for f in flows])
flows_df['overflow'] = flows_df['overflow'].map({True: '⚠️', False: ''})
display(flows_df)

,from_zone,to_shelter,to_shelter_id,people,overflow
0,中正區-南福里,臺北市中正運動中心,SA100-0006,483,
1,中正區-南福里,臺北市立成功高級中學,SA100-0010,1607,
2,松山區-龍田里,民生公園,SA105-1032,1554,
3,松山區-龍田里,敦化國小,SA105-1017,392,
4,信義區-安康里,臺北市立松山高級工農職業學校,SA110-0014,1108,
5,信義區-安康里,市府轉運站,SA110-0024,413,
6,信義區-安康里,臺北市立圖書館六合分館,SA110-0026,46,
7,信義區-安康里,臺北市信義區三興國民小學,SA110-0008,327,
8,松山區-復盛里,敦化國小,SA105-1017,530,
9,松山區-復盛里,臺北市立圖書館中崙分館,SA105-1027,37,


## Cell 6 — Shelter utilisation

In [ ]:
shelter_summary = pd.DataFrame([
    {
        'id':        s.id,
        'name':      s.name,
        'district':  s.district,
        'village':   s.village,
        'capacity':  s.capacity,
        'allocated': s.allocated,
        'over_cap':  max(0, s.allocated - s.capacity),
        'util%':     round(s.utilisation * 100, 1),
    }
    for s in shelters
    if s.allocated > 0
]).sort_values('allocated', ascending=False).reset_index(drop=True)

display(shelter_summary)

,id,name,district,village,capacity,allocated,over_cap,util%
0,SA100-0010,臺北市立成功高級中學,中正區,幸福里,2514,1607,0,63.9
1,SA105-1032,民生公園,松山區,介壽里,1554,1554,0,100.0
2,SA106-0025,忠孝新生站,大安區,民輝里,2803,1398,0,49.9
3,SA110-0001,臺北市信義區博愛國民小學,信義區,安康里,1363,1363,0,100.0
4,SA105-1024,南京復興站,松山區,中正里,1270,1270,0,100.0
5,SA110-0014,臺北市立松山高級工農職業學校,信義區,廣居里,1108,1108,0,100.0
6,SA106-0012,臺北市大安區古亭國民小學,大安區,古風里,981,981,0,100.0
7,SA110-0004,臺北市立興雅國民中學,信義區,安康里,962,962,0,100.0
8,SA105-1017,敦化國小,松山區,中正里,922,922,0,100.0
9,SA108-0014,臺北市萬華區萬大國民小學,萬華區,全德里,833,833,0,100.0


## Cell 7 — Save flows.json

In [ ]:
output = {
    'meta': {
        'total_zones':         len(impact_data),
        'total_shelters_used': len(shelter_summary),
        'total_allocated':     total_alloc,
        'total_overflow':      total_overflow,
    },
    'flows': [f.to_dict() for f in flows],
}

with open('flows.json', 'w', encoding='utf-8') as fp:
    json.dump(output, fp, ensure_ascii=False, indent=2)

print('Saved → flows.json')
print(json.dumps(output['meta'], indent=2, ensure_ascii=False))

Saved → flows.json
{
  "total_zones": 13,
  "total_shelters_used": 29,
  "total_allocated": 19454,
  "total_overflow": 0
}


## Cell 8 — Sanity check

In [ ]:
total_input = sum(int(z['預估避難人數']) for z in impact_data)
assert total_alloc == total_input, (
    f'Allocation mismatch: input={total_input}, allocated={total_alloc}'
)
print(' Sanity check passed — every refugee is accounted for.')

✅ Sanity check passed — every refugee is accounted for.
